Tabular Deep Learning: FT-Transformer, TabNet, PLE, and In-Context TabPFN

Deep learning architectures and feature selection mechanisms for tabular data:
1. Feature Tokenizer (FT-Transformer): Continuous affine projection and categorical token embeddings.
2. Sparsemax: Exact Euclidean projection onto the probability simplex for sparse attention.
3. TabNet Attentive Feature Selection: Sequential masking and prior scale recurrence.
4. Progressive Layered Extraction (PLE): Multi-task expert routing eliminating negative transfer.

## 2. Mathematical Formulations & Theoretical Foundations

- **FT-Transformer Numerical Tokenizer**:
  $$e_{\text{num}}^{(j)} = x_j \cdot W_j + b_j \in \mathbb{R}^D$$
- **Sparsemax Simplex Projection**:
  $$\operatorname{sparsemax}(z)_i = \max(z_i - \tau(z), 0), \quad \tau(z) = \frac{\sum_{j=1}^{k(z)} z_{(j)} - 1}{k(z)}$$
- **TabNet Attentive Mask & Prior Update**:
  $$M[i] = \operatorname{sparsemax}(P[i-1] \cdot h(a)), \quad P[i] = P[i-1] \cdot (\gamma - M[i])$$
- **PLE Gated Task Routing**:
  $$\hat{y}_k = \sum_{j \in \mathcal{E}_k \cup \mathcal{S}} \operatorname{softmax}(g_k)_j \cdot \text{Expert}_j(x)$$

In [ ]:
import math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

torch.manual_seed(42)
REPO_ROOT = next(p for p in Path.cwd().parents if (p / "Cargo.toml").exists()) if not (Path.cwd() / "Cargo.toml").exists() else Path.cwd()
GOLDEN_PATH = REPO_ROOT / "weights" / "07_domain_tasks" / "10_tabular_ft_transformer_tabnet_ple_and_tabpfn_in_context.safetensors"
golden = load_file(str(GOLDEN_PATH))
print(f"Loaded {len(golden)} tensors from golden checkpoint.")

In [ ]:
# 1. FT-Transformer Feature Tokenizer
def ft_transformer_tokenize(
    x_num: torch.Tensor,
    w_num: torch.Tensor,
    b_num: torch.Tensor,
    cat_tokens: torch.Tensor,
    cls_token: torch.Tensor,
) -> torch.Tensor:
    b, _ = x_num.shape
    num_tokens = x_num.unsqueeze(-1) * w_num.unsqueeze(0) + b_num.unsqueeze(0)
    cls_expanded = cls_token.expand(b, 1, -1)
    return torch.cat([cls_expanded, num_tokens, cat_tokens], dim=1)

tokens_calc = ft_transformer_tokenize(
    golden["x_num"],
    golden["w_num"],
    golden["b_num"],
    golden["cat_tokens"],
    golden["cls_token"],
)
diff_tokens = torch.max(torch.abs(tokens_calc - golden["tokenized_ref"])).item()
print(f"FT-Transformer Tokenizer max abs error: {diff_tokens:.2e}")
assert diff_tokens < 1e-5

In [ ]:
# 2. Sparsemax Simplex Projection
def sparsemax(z: torch.Tensor, dim: int = -1) -> torch.Tensor:
    z_sorted, _ = torch.sort(z, descending=True, dim=dim)
    d = z.shape[dim]
    range_d = torch.arange(1, d + 1, dtype=z.dtype, device=z.device)
    for _ in range(z.ndim - 1):
        range_d = range_d.unsqueeze(0)
    cumsum_z = torch.cumsum(z_sorted, dim=dim)
    condition = 1.0 + range_d * z_sorted > cumsum_z
    k_z = torch.sum(condition.to(z.dtype), dim=dim, keepdim=True)
    tau_z = (torch.gather(cumsum_z, dim, (k_z - 1).long()) - 1.0) / k_z
    return torch.clamp(z - tau_z, min=0.0)

sm_calc = sparsemax(golden["z_sparsemax"])
diff_sm = torch.max(torch.abs(sm_calc - golden["sparsemax_ref"])).item()
print(f"Sparsemax max abs error: {diff_sm:.2e}")
assert diff_sm < 1e-5

In [ ]:
# 3. TabNet Attentive Feature Mask & Prior Scale
def tabnet_attentive_mask(p_prior: torch.Tensor, h_features: torch.Tensor, gamma: float = 1.3):
    logits = p_prior * h_features
    mask = sparsemax(logits, dim=-1)
    p_next = p_prior * (gamma - mask)
    return mask, p_next

mask_calc, p_next_calc = tabnet_attentive_mask(golden["p_prior"], golden["h_feat"], gamma=1.3)
diff_mask = torch.max(torch.abs(mask_calc - golden["tabnet_mask_ref"])).item()
diff_p_next = torch.max(torch.abs(p_next_calc - golden["tabnet_p_next_ref"])).item()
print(f"TabNet Mask max abs error: {diff_mask:.2e}")
print(f"TabNet Prior Update max abs error: {diff_p_next:.2e}")
assert diff_mask < 1e-5
assert diff_p_next < 1e-5

In [ ]:
# 4. PLE Multi-Task Gated Routing & Summary Table
def ple_task_routing(
    expert_outputs: torch.Tensor,
    gate_logits: torch.Tensor,
    expert_indices: torch.Tensor,
) -> torch.Tensor:
    candidates = expert_outputs[:, expert_indices, :]
    weights = F.softmax(gate_logits, dim=-1).unsqueeze(-1)
    return torch.sum(candidates * weights, dim=1)

ple_calc = ple_task_routing(golden["expert_outputs"], golden["gate_logits"], golden["expert_indices"])
diff_ple = torch.max(torch.abs(ple_calc - golden["ple_routed_ref"])).item()

print("=" * 65)
print(f"{'Operator':<32} | {'Max Abs Error':<15} | Status")
print("-" * 65)
print(f"{'FT-Transformer Tokenizer':<32} | {diff_tokens:<15.2e} | PASSED")
print(f"{'Sparsemax Projection':<32} | {diff_sm:<15.2e} | PASSED")
print(f"{'TabNet Attentive Mask':<32} | {diff_mask:<15.2e} | PASSED")
print(f"{'TabNet Prior Scale Update':<32} | {diff_p_next:<15.2e} | PASSED")
print(f"{'PLE Multi-Task Routing':<32} | {diff_ple:<15.2e} | PASSED")
print("=" * 65)
assert diff_ple < 1e-5

## 8. Summary, Invariants & Diagnostic Checklist

- [x] FT-Transformer projects continuous tabular columns into uniform embedding spaces alongside categorical embeddings.
- [x] Sparsemax maps logits directly to exact zeros on the probability simplex, facilitating sparse feature attribution.
- [x] TabNet sequential masking dynamically shifts focus across features across iterative decision steps.
- [x] Progressive Layered Extraction separates task-specific from shared expert channels, preventing negative transfer.